# NYC Motor Vehicle Collisions — Exploratory Analysis
### nyc-crashes-exploration | Initial Data Wrangling

In [ ]:
## Import Libraries

import pandas as pd
import numpy as np


In [ ]:
## Load the Dataset
df = pd.read_csv('../data/nyc_crashes.csv')

print("Dataset loaded successfully!")


## First Look at the Data

In [ ]:
# Preview first few rows
df.head()


In [ ]:
# Check dimensions
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")


In [ ]:
# Data types and missing values overview
df.info()


### Explore Individual Columns

In [ ]:
# Statistical summary of numerical columns
df.describe()

In [ ]:
# clean names of column
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

In [ ]:
# Most common boroughs where crashes happen
df['borough'].value_counts()

In [ ]:
# Top contributing factors to crashes
df['contributing_factor_vehicle_1'].value_counts().head(10)


In [ ]:
# Most involved vehicle types
df['vehicle_type_code_1'].value_counts().head(10)


### Basic Cleaning

In [ ]:
### Check for Missing Values
# Count missing values per column
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Values': missing,
    'Percentage (%)': missing_pct
})

print(missing_summary[missing_summary['Missing Values'] > 0])



In [ ]:
### Check for Duplicates
print(f"Duplicate rows: {df.duplicated().sum()}")


In [ ]:
# Drop duplicates if any
df = df.drop_duplicates()
print(f"Shape after removing duplicates: {df.shape}")


In [ ]:
### Rename Columns for Clarity
df.rename(columns={
    'crash_date'                    : 'date',
    'crash_time'                    : 'time',
    'number_of_persons_injured'     : 'injured',
    'number_of_persons_killed'      : 'killed',
    'contributing_factor_vehicle_1' : 'main_cause',
    'vehicle_type_code1'            : 'vehicle_type',
    'collision_id'                  : 'id'
}, inplace=True)

print("Columns renamed successfully!")
df.head(2)


### Fix Data Types

In [ ]:
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'])


In [ ]:
# Ensure injured/killed are numeric
df['injured'] = pd.to_numeric(df['injured'], errors='coerce')
df['killed']  = pd.to_numeric(df['killed'],  errors='coerce')

print(df[['date', 'injured', 'killed']].dtypes)


### Drop Unnecessary Columns

In [ ]:
# Drop duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Duplicates removed: {before - len(df):,}  →  {len(df):,} rows remain")

In [ ]:
# Drop columns with too many missing values or low analytical value
cols_to_drop = [
    'location',
    'cross_street_name',
    'off_street_name',
    'contributing_factor_vehicle_3',
    'contributing_factor_vehicle_4',
    'contributing_factor_vehicle_5',
    'vehicle_type_code_3',
    'vehicle_type_code_4',
    'vehicle_type_code_5'
]


In [ ]:
# Only drop columns that actually exist in the df
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)

print(f"Remaining columns: {df.shape[1]}")
print(df.columns.tolist())


### Filtering & Sorting

In [ ]:
### Filter with Boolean Indexing — Fatal Crashes Only
fatal_crashes = df[df['killed'] > 0]
print(f"Fatal crashes: {len(fatal_crashes):,}")
fatal_crashes[['date', 'borough', 'killed', 'main_cause']].head(10)


In [ ]:
### Filter with .loc[] — Crashes in Brooklyn
brooklyn = df.loc[df['borough'] == 'BROOKLYN']
print(f"Crashes in Brooklyn: {len(brooklyn):,}")
brooklyn[['date', 'time', 'injured', 'main_cause']].head(5)


In [ ]:
### Filter with .iloc[] — First 5 rows, first 4 columns
df.iloc[0:5, 0:4]


In [ ]:
### Sort by Number of Injuries (Descending)
df.sort_values('injured', ascending=False)[
    ['date', 'borough', 'injured', 'killed', 'main_cause']
].head(10)


### Early Observations & Questions

In [ ]:
# Which borough has the most crashes?
print("Crashes by Borough:")
print(df['borough'].value_counts())


In [ ]:
# What is the most common cause of crashes?
print("\n Top 5 Causes:")
print(df['main_cause'].value_counts().head(5))


In [ ]:
# What percentage of crashes resulted in injuries?
pct_injured = (df['injured'] > 0).mean() * 100
print(f"\n Crashes with at least 1 injury: {pct_injured:.2f}%")


In [ ]:
# What percentage of crashes were fatal?
pct_fatal = (df['killed'] > 0).mean() * 100
print(f" Fatal crashes: {pct_fatal:.2f}%")


###  Summary

In [ ]:
print("=" * 45)
print("        PROJECT SUMMARY — NYC CRASHES")
print("=" * 45)
print(f"  Total records loaded   : {len(df):>10,}")
print(f"  Columns remaining      : {df.shape[1]:>10}")
print(f"  Date range             : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Fatal crashes          : {(df['killed'] > 0).sum():>10,}")
print(f"  Crashes with injuries  : {(df['injured'] > 0).sum():>10,}")
print("=" * 45)


## Cleaning + EDA + Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Plot style 
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.dpi": 130,
    "figure.figsize": (10, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

### Handle missing values

In [ ]:
# Borough: keep NaN but label them for aggregations
df['borough'] = df['borough'].fillna('UNKNOWN')